# 01. External crop-reference audit

The classifier needs traceable labels for carrot, cabbage and potato. This audit shows which sources were considered, which contributed to the saved matched experiment, and what can be learned from the reference files before imagery screening.

## Settings

Paths are relative to the repository. The three reference settings point to local releases for the candidate audits. The two parcel audits identify the saved experiment profiled in [Notebook 08](08_dataset_summary.ipynb). Update both together when choosing another experiment. The report folder receives a new audit directory for each run.

In [1]:
# Local reference releases used for the candidate audit.
cropharvest_labels = "data/external/cropharvest/labels.geojson"
eurocrops_folder = "data/external/eurocrops_v2"
eurocrops_mapping = "data/external/eurocrops_v2/eurocrops.csv"
usda_folder = "data/external/usda_cdl"

# Saved parcel records used to confirm which campaigns contributed fields.
development_audit = "pipeline_outputs/development_error_review_2026-11-19/parcel_diagnostics.json"
test_audit = "pipeline_outputs/test_audit_2026-11-19/parcel_audit.json"
report_folder = "pipeline_outputs/collection/reports"

In [2]:
from pathlib import Path
from tempfile import mkdtemp
import json
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))

from crop_pipeline.collection_summary import inspect_collection
from crop_pipeline.notebook_display import show_table
from crop_pipeline.output_layout import refresh
from crop_pipeline.sources.dataset_audit import audit_cropharvest, audit_eurocrops, audit_usda_cdl

report_root = root / report_folder
report_root.mkdir(parents=True, exist_ok=True)
audit_folder = Path(mkdtemp(prefix="reference_audit_", dir=report_root))

## Sources considered and their role in the experiment

The European development sources supplied traceable crop codes and parcel geometries or masks. JECAM supplied a separate test population that includes a tropical highland site in Madagascar. The supplements extend testing to other crop campaigns. The Paoay reference notes describe possible crops and rotations for application planning.

| Source or campaign | Decision | Reason |
|---|---|---|
| Sen4AgriNet | Used for development | Supplies carrot and potato under one labelling system in Spain and France. |
| FranceCrops | Used for development | Supplies explicit cabbage and table potato labels tied to parcel geometry. |
| EuroCrops Dutch carrot | Used for development | Supplies a second source of carrot parcels. |
| CropHarvest JECAM | Used as the primary external test | Covers all three target crops across Madagascar and Senegal. Madagascar provides the closest documented highland comparison in the selected campaigns. |
| CropHarvest non-JECAM campaigns | Used as supplemental tests | Extend the cabbage and carrot/potato checks to additional settings. |
| California DWR and Denmark Marker | Used as supplemental tests | Provide campaign-specific checks for carrot and cabbage respectively. |
| EuroCrops Dutch cabbage | Attempted test supplement, retired | The team reported insufficient clear-pixel support. |
| USDA Cropland Data Layer | Considered in the reference inventory | Supplies classified raster pixels. The experiment relies on identifiable parcel groups. |
| Paoay reference areas | Used for local context | Area boundaries and crop-calendar notes support field planning. Crop occupancy during the historical window remains unverified. |

The table below confirms which campaigns appear in the saved matched parcel records. Each row counts distinct parcel groups within one project split. [Source rationale and crop-label crosswalk](../../docs/classification_dataset_register.md)

In [3]:
development_path = root / development_audit
test_path = root / test_audit
if development_path.is_file() and test_path.is_file():
    development = json.loads(development_path.read_text())
    tests = json.loads(test_path.read_text())
    split_names = {"geographic_cv": "Training", "locked_validation": "Validation"}
    campaign_names = {
        "sen4agrinet": "Sen4AgriNet",
        "francecrops": "FranceCrops",
        "eurocrops": "EuroCrops Dutch carrot",
        "jecam": "CropHarvest JECAM",
        "non_jecam_cabbage_supplement_v1": "CropHarvest non-JECAM cabbage",
        "non_jecam_carrot_potato_supplement_v1": "CropHarvest non-JECAM carrot and potato",
        "california_dwr_carrot_2023_v1": "California DWR",
        "denmark_marker_cabbage_2023_v1": "Denmark Marker",
    }

    # Count each parcel group once within its source and split.
    groups = {}
    for row in development:
        if row["cohort"] == "fixed_matched" and row["stage"] in split_names:
            key = (split_names[row["stage"]], row["source"])
            groups.setdefault(key, set()).add(row["partition_group"])
    for row in tests:
        if row["cohort"] == "fixed_matched":
            groups.setdefault(("External test", row["scope"]), set()).add(row["group"])

    show_table([
        {"Project split": split, "Campaign": campaign_names.get(source, source),
         "Matched parcels": len(parcels)}
        for (split, source), parcels in sorted(groups.items())
    ], title="Campaign participation in the saved matched experiment")
else:
    print("Matched campaign counts are unavailable. Check the two saved parcel audit paths.")

Project split,Campaign,Matched parcels
External test,California DWR,62
External test,Denmark Marker,15
External test,CropHarvest JECAM,83
External test,CropHarvest non-JECAM cabbage,5
External test,CropHarvest non-JECAM carrot and potato,16
Training,EuroCrops Dutch carrot,47
Training,FranceCrops,61
Training,Sen4AgriNet,91
Validation,EuroCrops Dutch carrot,11
Validation,FranceCrops,21


## What the reference releases contain

The next three checks describe candidate evidence before the project applies imagery and window rules. CropHarvest counts label records and screened parcels. EuroCrops counts distinct declarations. USDA CDL counts classified pixels. Interpret each count in its own unit.

### CropHarvest field labels

The audit checks the pinned label release and identifies screened parcels with at least one candidate interior 10 m pixel. The source table retains the upstream test flag, which describes the original release. The project splits appear in the matched campaign table above. [Pinned CropHarvest label release](https://zenodo.org/records/10251170)

In [4]:
labels = root / cropharvest_labels
if labels.is_file():
    total, counts, by_source = audit_cropharvest(labels, audit_folder / "cropharvest")
    print(f"Label records in the full release · {total:,}")
    show_table(counts.rename_axis("Crop").reset_index(), title="CropHarvest candidate screening")
    show_table(by_source, title="Interior-pixel candidates by source and upstream split")
else:
    print(f"CropHarvest audit unavailable. Expected {cropharvest_labels}.")

Label records in the full release · 113,893


Crop,Candidates,Passed,Excluded,Raw alias matches,Interior pixel parcels
Carrot,522,522,0,622,105
Cabbage,190,190,0,265,91
Potato,670,670,0,780,356


Crop,Source,Upstream test,Interior pixel parcels
Cabbage,ile-de-france,No,14
Cabbage,jecam,No,56
Cabbage,martinique-france,No,16
Cabbage,reunion-france,No,5
Carrot,ile-de-france,No,27
Carrot,jecam,No,73
Carrot,martinique-france,No,3
Carrot,reunion-france,No,2
Potato,ile-de-france,No,169
Potato,jecam,No,92


### EuroCrops annual declarations

The inventory uses the translated crop names in the mapping file. A zero means the audit found no matches under its supported names. The Dutch carrot campaign uses explicit national production codes, documented in the crosswalk. [EuroCrops release](https://jeodpp.jrc.ec.europa.eu/ftp/public/JRC-OpenData/DRLL/EuroCropsV2/)

In [5]:
parquet_files = sorted((root / eurocrops_folder).rglob("*.parquet"))
mapping = root / eurocrops_mapping
if mapping.is_file() and parquet_files:
    counts = audit_eurocrops(parquet_files, mapping)
    if counts.empty:
        print("No matching annual GeoParquet filenames found. Expected names such as NL_2021.parquet.")
    else:
        declarations = counts.pivot(index=["region", "year"], columns="crop", values="declared_parcels")
        show_table(declarations.reset_index(), title="Distinct declared EuroCrops parcels")
else:
    print("EuroCrops audit unavailable. Check the annual files and crop-code mapping paths.")

crop,Region,Year,Cabbage,Carrot,Potato
,NL,"2,021",0,0,0


### USDA Cropland Data Layer

This exploratory inventory records the number of unmasked raster pixels assigned to each crop code. Parcel counts would require independent field boundaries. [USDA CDL class definitions](https://developers.google.com/earth-engine/datasets/catalog/USDA_NASS_CDL)

In [6]:
usda_root = root / usda_folder
rasters = sorted(path for path in usda_root.rglob("*") if path.suffix.lower() in {".tif", ".tiff"})
if rasters:
    pixel_counts = []
    for raster in rasters:
        counts = audit_usda_cdl(raster)
        pixel_counts.append({"Raster": str(raster.relative_to(usda_root)), **counts.to_dict()})
    show_table(pixel_counts, title="USDA CDL classified pixels per raster")
else:
    print(f"USDA CDL audit unavailable. Expected GeoTIFF files in {usda_folder}.")

Raster,Carrot,Cabbage,Potato
CDL_2023_06.tif,"126,081","15,565","67,421"


## Collection inventory and handoff

The saved collection ledger shows which parcel groups were selected for collection. A selected group can still fail imagery, overlap or window screening. [Notebook 08 — Classification dataset profile](08_dataset_summary.ipynb) follows those groups into the usable experiment and shows their crop-label mappings, geographic distribution, class balance and timing relative to the imagery. It also maps the local Paoay reference areas.

In [7]:
saved_collection = inspect_collection(root)
show_table(saved_collection["profile"], title="Saved collection selections by source, crop and partition")
refresh(root)
print(f"Saved metadata issues · {len(saved_collection['issues'])}")
print(f"Reference audit folder · {audit_folder.relative_to(root)}")

Source,Crop,Partition,Role,Selected parcel groups
cropharvest,Cabbage,Reserved test,external_test,189
cropharvest,Carrot,Reserved test,external_test,520
cropharvest,Potato,Reserved test,external_test,669
eurocrops,Carrot,Reserved test,reserved_development,435
eurocrops,Carrot,Training,development,"1,321"
eurocrops,Carrot,Validation,development,438
francecrops,Cabbage,Reserved test,reserved_development,348
francecrops,Cabbage,Training,development,"1,111"
francecrops,Cabbage,Validation,development,341
francecrops,Potato,Reserved test,reserved_development,23


Saved metadata issues · 1
Reference audit folder · pipeline_outputs/collection/reports/reference_audit_jddt_vux
